# Create new Salesforce Job__c

POST a new `Job__c` record from Supabase `job_current`.

**Test markers:** Set `PROXI_SF_TEST_MODE=true` in `.env` to apply `[TEST]` / `TEST-` / `[TEST RECORD]` prefixes on the payload (safe default for manual testing). Leave unset for **go-live-shaped** creates.

- Set **`SUPABASE_JOB_ID`** below, then run all cells.
- `DRY_RUN = True` previews the payload without writing.

In [1]:
import os, sys, json
from pathlib import Path

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")
print("Ready")

Ready


In [ ]:
# --- Change these ---
SUPABASE_JOB_ID = "19596"
SUPABASE_SCHEMA = "public"
DRY_RUN = False

In [3]:
from utils.supabase_db import load_job_current_row_for_salesforce

job_row = load_job_current_row_for_salesforce(SUPABASE_JOB_ID, schema=SUPABASE_SCHEMA)
print(f"Loaded job_id={job_row.get('job_id')}  city={job_row.get('city')}  state={job_row.get('state')}")

Loaded job_id=19587  city=Escanaba  state=MI


In [4]:
from utils.salesforce import get_token_auto

token = get_token_auto(
    os.environ["SALESFORCE_CONSUMER_KEY"],
    os.environ["SALESFORCE_CONSUMER_SECRET"],
    os.environ.get("SALESFORCE_USERNAME") or None,
    os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes"),
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes"),
    token_url=os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com",
)
INSTANCE_URL = token["instance_url"]
ACCESS_TOKEN = token["access_token"]
print("Authenticated:", INSTANCE_URL)

Authenticated: https://proxi.my.salesforce.com


In [5]:
from utils.sf_job_payload import prepare_payload_for_write
from utils.sf_job_rest_minimal import describe_sobject

JOB_OBJECT = os.environ.get("SALESFORCE_JOB_OBJECT", "Job__c").strip()
describe = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT)

fields = prepare_payload_for_write(
    job_row,
    describe,
    use_canonical_description=True,
    for_update=False,
    description_use_html=True,
)

# Optional test-only mutations (production pipeline uses the same flag in PATCH sync).
if os.environ.get("PROXI_SF_TEST_MODE", "").lower() in ("1", "true", "yes"):
    TEST_PREFIX = "[TEST] "
    if "Name" in fields:
        fields["Name"] = TEST_PREFIX + (fields["Name"] or "")
    if "External_Job_ID__c" in fields:
        fields["External_Job_ID__c"] = "TEST-" + (fields.get("External_Job_ID__c") or "")
    if "Job_Client_Job_Description__c" in fields:
        fields["Job_Client_Job_Description__c"] = "[TEST RECORD] " + (fields["Job_Client_Job_Description__c"] or "")
else:
    print("PROXI_SF_TEST_MODE not enabled — payload is go-live shaped (no TEST prefixes).")

print(f"{len(fields)} fields:", sorted(fields.keys()))

Skipped (not createable on object): Job_Facility_Display__c, Job_Worksite_1_Address__c, Job_Point_of_Contact__c, Job_Standard_Schedule__c, Job_Provider_Start_Date__c, Job_Provider_End_Date__c, Position_Type_DJC__c, Specialty_DJC__c, Occupation_DJC__c, Worksite_Parent__c
18 fields: ['External_Job_ID__c', 'External_Job_Link__c', 'Insight__c', 'Job_Account__c', 'Job_City__c', 'Job_Client_Job_Description__c', 'Job_Client_Job_Id__c', 'Job_Dates_Needed__c', 'Job_Patient_Ages__c', 'Job_Ranking__c', 'Job_Recruitment_Level__c', 'Job_State__c', 'Job_Status__c', 'Job_Support_Staff__c', 'Job_Types_of_Cases__c', 'Job_Volume__c', 'Job_Worksite_Location_1__c', 'Salary_Pay_Range__c']


Note: Job_Recruitment_Level__c value 'Critical' not allowed; using first active picklist value 'Building Roster'


In [6]:
from utils.sf_job_rest_minimal import create_job_record

show = dict(fields)
dk = "Job_Client_Job_Description__c"
if dk in show and len(str(show[dk])) > 500:
    show[dk] = str(show[dk])[:500] + f"... ({len(str(fields[dk]))} chars)"
print(json.dumps(show, indent=2, default=str))

if DRY_RUN:
    print("\nDRY_RUN — set DRY_RUN = False in cell 2 and re-run from there.")
else:
    result = create_job_record(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT, fields)
    new_id = result.get("id", "(unknown)")
    print(f"\nCreated {JOB_OBJECT} → {new_id}")

{
  "External_Job_ID__c": "TEST-19587",
  "Job_Account__c": "0015f00000HH63kAAD",
  "Job_Worksite_Location_1__c": "0015f00000S30EhAAJ",
  "Job_Client_Job_Id__c": "3159 - Escanaba, MI",
  "Job_Client_Job_Description__c": "[TEST RECORD] <p><strong>General Dentist Locum Tenens Opportunity in Escanaba, MI</strong></p><p><br/></p><p>We are seeking a General Dentist for a locum tenens opportunity in Escanaba, Michigan. This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><br/></p><p>This role is ideal for a dentist comfortable with surgical extractions and dentures who enjoys working in a collaborative environm... (1695 chars)",
  "External_Job_Link__c": "https://portal.kimedics.com/app/workspace/job-posts/19587",
  "Job_Status__c": "Closed",
  "Job_State__c": "Michigan",
  "Job_City__c": "Escanaba",
  "Insight__c": "*Must have active MI DEA with all schedules and CSR at time of submission\n**Must be r